# Quantum Face Detection Simulator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NAVTEJJ/face_detection_benchmark/blob/main/notebooks/quantum_face_simulator.ipynb)

An interactive version of the face detection benchmark: a classical CNN against a **4-qubit quanvolutional network (QNN)**, on LFW faces vs CIFAR-10 non-faces (32×32 grayscale).

**How to use:** run all cells (*Run → Run All Cells*), then use the controls under each section. Nothing trains here: the five models were trained once with the study's exact settings (seed 0, `train_checkpoints.py`) and are loaded from `checkpoints/`.

| Section | What you can do |
|---|---|
| 1. The circuit | See the 4-qubit circuit every 2×2 image patch goes through |
| 2. Circuit simulator | Set the 4 pixel values, pick random or trained weights, choose the number of measurement shots, and watch the quantum state and outputs |
| 3. Quanvolution viewer | Pick any image and see the feature maps made by the quantum layer and by a classical layer of the same shape |
| 4. Live detector | Classify any test image with all five models; add brightness, contrast and noise changes |
| 5. Deployment simulator | Change how much background there is per face and the decision threshold; see precision and false alarms |
| 6. Study results | The controlled 5-seed comparison behind the conclusions |

In [ ]:
# Setup. On Google Colab this also fetches the repository and installs PennyLane (about a minute).
import os, sys
if "google.colab" in sys.modules and not os.path.exists("face_detection_benchmark"):
    !git clone -q https://github.com/NAVTEJJ/face_detection_benchmark.git
    %pip install -q pennylane
ROOT = next(p for p in ["face_detection_benchmark", "..", "."]
            if os.path.exists(os.path.join(p, "src", "demo.py")))
os.chdir(ROOT)
sys.path.insert(0, os.getcwd())

import warnings; warnings.filterwarnings("ignore")
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, HTML, Image
from src import demo
%matplotlib inline
plt.rcParams.update({"figure.facecolor": "white", "axes.spines.top": False, "axes.spines.right": False})

data = demo.load_demo_data(seed=0)       # LFW + CIFAR-10, downloaded automatically the first time
models = demo.load_models()               # the five trained models
scores = np.load("checkpoints/scores.npz")

faces_test = data["X_test"][data["y_test"] == 1]
nonfaces_test = data["X_test"][data["y_test"] == 0]
W_TRAINED = models["qnn_trained"].weights.detach().numpy()
W_RANDOM = models["qnn_frozen"].weights.detach().numpy()
print(f"Test set: {len(faces_test)} faces, {len(nonfaces_test)} non-faces | "
      f"background pool: {len(data['background'])} | models: {', '.join(models)}")

## 1. The quantum circuit

Each 2×2 patch of the image (4 pixel values) goes through this circuit:

1. **Encoding:** pixel value $x_i \in [0,1]$ sets qubit $i$ with a rotation $R_Y(\pi x_i)$.
2. **Two layers:** each qubit gets $R_X$, $R_Y$, $R_Z$ rotations (12 angles per layer, 24 in total), then a ring of CNOTs (0→1→2→3→0) entangles neighbouring qubits.
3. **Readout:** the expectation $\langle Z \rangle \in [-1, 1]$ of every qubit becomes one output channel.

Sliding over the 32×32 image with stride 2 gives **4 feature maps of 16×16**: the quantum version of a convolution layer. Below are the trained weights (seed 0), with an example patch.

In [ ]:
fig = demo.draw_circuit([0.2, 0.8, 0.5, 0.9], W_TRAINED)
plt.show()

## 2. Circuit simulator

Set the four pixel values of a patch and pick which weights to use. The simulator computes the full **quantum state**, 16 probabilities for the 16 basis states $|q_0 q_1 q_2 q_3\rangle$, and the four $\langle Z \rangle$ outputs.

**Shots** mimic real hardware: a quantum computer only returns measurement samples, so each $\langle Z\rangle$ has to be estimated from a finite number of repetitions. With few shots the estimate is noisy; *exact* is the ideal simulator value.

In [ ]:
px = [W.FloatSlider(value=v, min=0, max=1, step=0.01, description=f"pixel {i}", continuous_update=False)
      for i, v in enumerate([0.2, 0.8, 0.5, 0.9])]
w_choice = W.ToggleButtons(options=["trained", "random"], description="weights")
shots = W.SelectionSlider(options=["exact", 10, 100, 1000, 10000], value=100, description="shots",
                          continuous_update=False)
reshoot = W.Button(description="measure again", icon="refresh")
trial = W.IntText(value=0, layout=W.Layout(display="none"))   # bumped by the button to redraw
reshoot.on_click(lambda _: setattr(trial, "value", trial.value + 1))
rng = np.random.default_rng(0)

def show_circuit(p0, p1, p2, p3, which, n_shots, _trial):
    pixels = [p0, p1, p2, p3]
    probs, z, bits = demo.circuit_state(pixels, W_TRAINED if which == "trained" else W_RANDOM)
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6), gridspec_kw={"width_ratios": [1, 3.6, 1.8]})

    ax[0].imshow(np.array(pixels).reshape(2, 2), cmap="gray", vmin=0, vmax=1)
    for k, v in enumerate(pixels):
        ax[0].text(k % 2, k // 2, f"{v:.2f}", ha="center", va="center", color="red", fontsize=11)
    ax[0].set_title("2×2 input patch"); ax[0].set_xticks([]); ax[0].set_yticks([])

    labels = ["".join(map(str, b)) for b in bits]
    ax[1].bar(range(16), probs, color="#E8543A", alpha=0.85, label="exact probability")
    if n_shots != "exact":
        zs, counts = demo.sample_z(probs, bits, int(n_shots), rng)
        ax[1].scatter(range(16), counts / int(n_shots), color="black", s=18, zorder=3,
                      label=f"measured frequency ({n_shots} shots)")
    ax[1].set_xticks(range(16), labels, rotation=90, fontsize=8)
    ax[1].set_ylabel("probability"); ax[1].set_title("Quantum state: basis states |q0 q1 q2 q3>")
    ax[1].legend(fontsize=8, loc="upper right")

    ax[2].barh(range(4), z[::-1], color="#4A90D9", label="exact")
    if n_shots != "exact":
        ax[2].scatter(zs[::-1], range(4), color="black", s=30, zorder=3, label="from shots")
    ax[2].axvline(0, color="#999", lw=1)
    ax[2].set_yticks(range(4), [f"qubit {q}" for q in range(3, -1, -1)])
    ax[2].set_xlim(-1, 1); ax[2].set_title("Outputs <Z>"); ax[2].legend(fontsize=8)
    plt.tight_layout(); plt.show()
    print("exact <Z>:  " + "  ".join(f"q{i}={v:+.3f}" for i, v in enumerate(z)))
    if n_shots != "exact":
        print(f"from {n_shots} shots: " + "  ".join(f"q{i}={v:+.3f}" for i, v in enumerate(zs))
              + f"   (max error {np.abs(zs - z).max():.3f})")

out = W.interactive_output(show_circuit, {"p0": px[0], "p1": px[1], "p2": px[2], "p3": px[3],
                                          "which": w_choice, "n_shots": shots, "_trial": trial})
display(W.VBox([W.HBox(px[:2]), W.HBox(px[2:]), W.HBox([w_choice, shots, reshoot]), out]))

## 3. Quanvolution viewer

Pick an image. Each row shows what one front-end layer turns it into: the quantum circuit with **random** weights, the quantum circuit **trained** end-to-end, and the **classical** layer with exactly the same shape (2×2 kernel, stride 2, 4 channels). Every map is 16×16, with values in [−1, 1].

In [ ]:
cls = W.ToggleButtons(options=["face", "non-face"], description="class")
idx = W.IntSlider(value=0, min=0, max=len(faces_test) - 1, description="image #", continuous_update=False)
cls.observe(lambda c: idx.set_trait("max", (len(faces_test) if c["new"] == "face" else len(nonfaces_test)) - 1),
            names="value")

def show_maps(which, i):
    img = (faces_test if which == "face" else nonfaces_test)[i, 0]
    rows = [("qnn_frozen", "QNN, random circuit"), ("qnn_trained", "QNN, trained circuit"),
            ("cnn_matched", "Classical, same shape (trained)")]
    fig, ax = plt.subplots(3, 5, figsize=(13, 8))
    for r, (arm, name) in enumerate(rows):
        ax[r, 0].imshow(img, cmap="gray", vmin=0, vmax=1)
        ax[r, 0].set_ylabel(name, fontsize=10)
        maps = demo.front_end_maps(models[arm], arm, img)
        for c in range(4):
            ax[r, c + 1].imshow(maps[c], cmap="RdBu_r", vmin=-1, vmax=1)
            if r == 0:
                ax[r, c + 1].set_title(f"channel {c}" + (" (qubit %d)" % c))
    for a in ax.ravel():
        a.set_xticks([]); a.set_yticks([])
    ax[0, 0].set_title("input (32×32)")
    plt.suptitle(f"{which} #{i}: four 16×16 feature maps per layer (red = +1, blue = −1)", y=1.0)
    plt.tight_layout(); plt.show()

display(W.VBox([W.HBox([cls, idx]), W.interactive_output(show_maps, {"which": cls, "i": idx})]))

## 4. Live detector

All five models classify the selected test image. Change brightness, contrast and noise to see how robust each one is. The bars show each model's probability that the image is a face; above 0.5 counts as *face*.

In [ ]:
cls4 = W.ToggleButtons(options=["face", "non-face"], description="class")
idx4 = W.IntSlider(value=5, min=0, max=len(faces_test) - 1, description="image #", continuous_update=False)
cls4.observe(lambda c: idx4.set_trait("max", (len(faces_test) if c["new"] == "face" else len(nonfaces_test)) - 1),
             names="value")
bright = W.FloatSlider(value=0, min=-0.4, max=0.4, step=0.05, description="brightness", continuous_update=False)
contrast = W.FloatSlider(value=1, min=0.3, max=1.7, step=0.1, description="contrast", continuous_update=False)
noise = W.FloatSlider(value=0, min=0, max=0.3, step=0.02, description="noise", continuous_update=False)

def detect(which, i, b, c, n):
    img = (faces_test if which == "face" else nonfaces_test)[i, 0]
    x = np.clip((img - img.mean()) * c + img.mean() + b
                + np.random.default_rng(i).normal(0, n, img.shape), 0, 1).astype(np.float32)
    p = {arm: float(demo.predict_proba(models[arm], x[None, None])[0]) for arm in demo.ARMS}

    fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1, 2.6]})
    ax[0].imshow(x, cmap="gray", vmin=0, vmax=1); ax[0].set_xticks([]); ax[0].set_yticks([])
    ax[0].set_title(f"true label: {which.upper()}")
    names = [demo.LABELS[a] for a in demo.ARMS][::-1]
    vals = [p[a] for a in demo.ARMS][::-1]
    ax[1].barh(names, vals, color=[demo.COLORS[a] for a in demo.ARMS][::-1])
    ax[1].axvline(0.5, color="#555", ls="--", lw=1)
    for k, v in enumerate(vals):
        verdict = "face" if v >= 0.5 else "non-face"
        ok = (verdict == which)
        ax[1].text(min(v, 0.80) + 0.02, k, f"{v:.3f} → {verdict} {'✓' if ok else '✗'}", va="center", fontsize=9)
    ax[1].set_xlim(0, 1); ax[1].set_xlabel("P(face)")
    plt.tight_layout(); plt.show()

display(W.VBox([W.HBox([cls4, idx4]), W.HBox([bright, contrast, noise]),
                W.interactive_output(detect, {"which": cls4, "i": idx4, "b": bright, "c": contrast, "n": noise})]))

## 5. Deployment simulator

A real detector scans a photo window by window, so it sees far more background than faces. Set **how many background windows there are per face** and the **decision threshold**.

For each model, recall is measured on the test faces and the false-positive rate on 6,060 background images that none of the models trained on. Precision at a given ratio $r$ then follows directly:

$$\text{precision} = \frac{\text{recall}}{\text{recall} + r \cdot \text{FPR}}$$

Notice that ROC-AUC would not move at all here; precision does.

In [ ]:
ratio = W.IntSlider(value=10, min=1, max=100, description="bg per face", continuous_update=False)
thr = W.FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05, description="threshold", continuous_update=False)
y_t = scores["y_test"]

def deploy(r, t):
    rows = {a: demo.deployment_metrics(scores[f"{a}_test"][y_t == 1], scores[f"{a}_bg"], r, t) for a in demo.ARMS}
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
    names = [demo.LABELS[a] for a in demo.ARMS][::-1]
    cols = [demo.COLORS[a] for a in demo.ARMS][::-1]
    prec = [rows[a]["precision"] for a in demo.ARMS][::-1]
    fa = [rows[a]["false_alarms_per_100_faces"] for a in demo.ARMS][::-1]
    ax[0].barh(names, prec, color=cols); ax[0].set_xlim(0, 1); ax[0].set_title(f"Precision at 1:{r}")
    for k, v in enumerate(prec): ax[0].text(v + 0.01, k, f"{v:.3f}", va="center", fontsize=9)
    ax[1].barh(names, fa, color=cols); ax[1].set_title("False alarms per 100 real faces")
    ax[1].set_yticks([])
    for k, v in enumerate(fa): ax[1].text(v, k, f" {v:.0f}", va="center", fontsize=9)
    plt.tight_layout(); plt.show()
    html = "<table><tr><th>model</th><th>recall</th><th>false-positive rate</th><th>precision</th><th>false alarms / 100 faces</th></tr>"
    for a in demo.ARMS:
        m = rows[a]
        html += (f"<tr><td>{demo.LABELS[a]}</td><td>{m['recall']:.3f}</td><td>{m['fpr']:.4f}</td>"
                 f"<td><b>{m['precision']:.3f}</b></td><td>{m['false_alarms_per_100_faces']:.0f}</td></tr>")
    display(HTML(html + "</table>"))

display(W.VBox([W.HBox([ratio, thr]), W.interactive_output(deploy, {"r": ratio, "t": thr})]))

## 6. Study results: is the quantum layer actually better?

The sections above use one trained copy of each model. To compare fairly, `study.py` trained every model on **5 different data splits and initialisations** with the same budget (Adam, 12 epochs, best epoch by validation loss), and compared them **paired on the same splits**. "No clear difference" means the 95% confidence interval of the paired difference includes zero.

In [ ]:
s = json.load(open("outputs/study_summary.json"))
html = ("<table><tr><th>model</th><th>trainable params</th><th>balanced accuracy</th>"
        "<th>1:10 precision</th><th>1:10 average precision</th></tr>")
for arm, e in s["arms"].items():
    f = lambda k: f"{e[k]['mean']:.3f} ± {e[k]['ci95']:.3f}"
    html += (f"<tr><td>{demo.LABELS[arm]}</td><td>{e['trainable_params']:,}</td><td>{f('balanced.accuracy')}</td>"
             f"<td>{f('imbalanced.precision')}</td><td>{f('imbalanced.avg_precision')}</td></tr>")
html += "</table><br><table><tr><th>paired comparison (first − second)</th><th>balanced accuracy</th><th>1:10 precision</th><th>p (accuracy)</th></tr>"
for c in s["contrasts"]:
    a, p = c["balanced.accuracy"], c["imbalanced.precision"]
    clear = not (a["mean_diff"] - a["ci95"] <= 0 <= a["mean_diff"] + a["ci95"])
    html += (f"<tr><td>{c['question']}</td><td>{a['mean_diff']:+.3f} ± {a['ci95']:.3f}"
             f"{' <b>(clear)</b>' if clear else ' (no clear difference)'}</td>"
             f"<td>{p['mean_diff']:+.3f} ± {p['ci95']:.3f}</td><td>{a['p']:.3f}</td></tr>")
display(HTML(html + "</table>"))
display(Image("outputs/fig_study.png"))

### Conclusion

- A quantum layer and a classical layer **of the same shape** perform the same, whether both are left random or both are trained.
- Training the circuit helps it, by about as much as training helps the classical layer.
- The big gap in the original single comparison came from the original CNN's much larger classifier head, not from anything quantum.
- All quantum results come from an exact, noise-free simulation. Section 2's *shots* control shows the first thing real hardware adds: sampling noise.

Code, method and every number: [github.com/NAVTEJJ/face_detection_benchmark](https://github.com/NAVTEJJ/face_detection_benchmark) · results page: [navtejj.github.io/face_detection_benchmark](https://navtejj.github.io/face_detection_benchmark/PRESENTATION.html)